In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
from sklearn.model_selection import train_test_split

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('Imports OK')

In [ ]:
SEED     = 42
N_TRIALS = 15
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = Path('../../')
DATA_DIR = BASE_DIR / 'data/processed'

matplotlib.rcParams.update({'figure.figsize': (9, 6), 'font.size': 12,
                            'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

X_raw       = df[feature_cols].values
y           = df['TARGET'].values
X_test_raw  = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

print(f'Train: {X_raw.shape}  | Default rate: {y.mean():.2%}')
print(f'Test : {X_test_raw.shape}')

---
## Preprocesamiento

Las redes neuronales requieren:
1. **Imputación** de valores faltantes (no manejan NaN nativamente)
2. **Escalado** (StandardScaler) para que los gradientes converjan

In [ ]:
# Imputación + escalado
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_imp      = imputer.fit_transform(X_raw)
X_scaled   = scaler.fit_transform(X_imp)

X_test_imp    = imputer.transform(X_test_raw)
X_test_scaled = scaler.transform(X_test_imp)

print(f'X_scaled  : shape={X_scaled.shape}  NaNs={np.isnan(X_scaled).sum()}')
print(f'X_test_sc : shape={X_test_scaled.shape}  NaNs={np.isnan(X_test_scaled).sum()}')

# Split train/val estratificado para Optuna
X_tr, X_val, y_tr, y_val = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=SEED
)
print(f'Train: {X_tr.shape} | Val: {X_val.shape}')

# Class weights para imbalance
class_weight = {0: 1.0, 1: float((y == 0).sum()) / float((y == 1).sum())}
print(f'class_weight: {class_weight}')

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc,4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec,4), Precision=round(prec,4), F1=round(f1,4))

def plot_roc_curve(y_true, y_prob, label, color, ax, title='ROC Curve'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC = {auc:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(title); ax.legend()
    return ax

print('Funciones definidas OK')

---
## Arquitectura MLP

**Estructura**: Input → [Dense(units, ReLU) → BatchNorm → Dropout] × n_layers → Dense(1, sigmoid)

**Optuna** busca: `n_layers` ∈ [1,3], `units` ∈ [32,256], `dropout_rate` ∈ [0.1,0.5], `learning_rate` ∈ [1e-4,1e-2]

In [ ]:
def build_model(n_layers, units, dropout_rate, learning_rate, input_dim):
    """MLP con BatchNorm + Dropout. Salida sigmoide para clasificación binaria."""
    inputs = tf.keras.Input(shape=(input_dim,))
    x = inputs
    for _ in range(n_layers):
        x = tf.keras.layers.Dense(units, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc')]
    )
    return model

INPUT_DIM = X_scaled.shape[1]
print(f'Input dim: {INPUT_DIM}')

# Test rápido del modelo
_test_model = build_model(2, 64, 0.3, 1e-3, INPUT_DIM)
print(_test_model.summary())

In [ ]:
def objective(trial):
    tf.random.set_seed(SEED + trial.number)

    n_layers      = trial.suggest_int('n_layers', 1, 3)
    units         = trial.suggest_categorical('units', [32, 64, 128, 256])
    dropout_rate  = trial.suggest_float('dropout_rate', 0.1, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size    = trial.suggest_categorical('batch_size', [256, 512, 1024])

    model = build_model(n_layers, units, dropout_rate, learning_rate, INPUT_DIM)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', patience=10, mode='max',
            restore_best_weights=True
        )
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=batch_size,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=0,
    )

    val_auc   = max(history.history['val_auc'])
    train_auc = history.history['auc'][np.argmax(history.history['val_auc'])]

    trial.set_user_attr('val_auc',       val_auc)
    trial.set_user_attr('train_auc',     train_auc)
    trial.set_user_attr('best_epoch',    int(np.argmax(history.history['val_auc'])) + 1)

    gap = max(0, train_auc - val_auc)
    return val_auc - 0.5 * gap

print(f'Lanzando Optuna — {N_TRIALS} trials...')
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f'\nMejores hiperparámetros:')
for k, v in best.params.items():
    print(f'  {k:<16}: {v}')
print(f'  best_epoch     : {best.user_attrs["best_epoch"]}')
print(f'\nObjetivo (Val AUC penalizado): {best.value:.4f}')
print(f'Val AUC   : {best.user_attrs["val_auc"]:.4f}')
print(f'Train AUC : {best.user_attrs["train_auc"]:.4f}')

In [ ]:
# Refit final en todo el train con mejores hiperparámetros
tf.random.set_seed(SEED)

best_p = study.best_trial.params

model_final = build_model(
    best_p['n_layers'], best_p['units'],
    best_p['dropout_rate'], best_p['learning_rate'],
    INPUT_DIM
)

callbacks_final = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=15, mode='max',
        restore_best_weights=True
    )
]

history_final = model_final.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=best_p['batch_size'],
    class_weight=class_weight,
    callbacks=callbacks_final,
    verbose=1,
)

y_prob_val  = model_final.predict(X_val, verbose=0).ravel()
y_prob_full = model_final.predict(X_scaled, verbose=0).ravel()

val_auc_final  = roc_auc_score(y_val,  y_prob_val)
train_auc_full = roc_auc_score(y,      y_prob_full)

metrics = compute_metrics(y, y_prob_full, threshold=0.5, label='Neural Network')
metrics['Val_AUC']   = round(val_auc_final, 4)
metrics['n_epochs']  = int(np.argmax(history_final.history['val_auc'])) + 1

print('\nRED NEURONAL — MÉTRICAS:')
for k, v in metrics.items():
    if k != 'Model':
        print(f'  {k:<14}: {v}')

---
## Visualizaciones

In [ ]:
# Optuna: historial de trials
trial_df = study.trials_dataframe()[['number','value']].rename(columns={'value':'Objetivo'})

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(trial_df['number'], trial_df['Objetivo'], 'o-', color='#9b59b6')
ax.axhline(study.best_value, color='#e74c3c', ls='--',
           label=f'Best = {study.best_value:.4f} (trial {study.best_trial.number})')
ax.set_xlabel('Trial'); ax.set_ylabel('Val AUC penalizado')
ax.set_title('Optuna — Red Neuronal (MLP)'); ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Learning curve (Train AUC vs Val AUC por época)
epochs_range = range(1, len(history_final.history['auc']) + 1)
best_ep = metrics['n_epochs']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, history_final.history['auc'],     color='#3498db', lw=1.5, alpha=0.8, label='Train AUC')
ax.plot(epochs_range, history_final.history['val_auc'], color='#e74c3c', lw=1.5, label='Val AUC')
ax.axvline(best_ep, color='gray', ls='--', alpha=0.7, label=f'Best epoch = {best_ep}')
ax.set_xlabel('Época'); ax.set_ylabel('AUC')
ax.set_title('Learning Curve — Red Neuronal (MLP)'); ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
plot_roc_curve(y, y_prob_full, label='Neural Network (MLP)', color='#9b59b6', ax=ax,
               title='ROC Curve — Red Neuronal (MLP)')
plt.tight_layout()
plt.show()

In [ ]:
# Score distribution TARGET=0 vs TARGET=1
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y_prob_full[y == 0], bins=60, alpha=0.6, color='#3498db', label='TARGET=0 (paga)', density=True)
ax.hist(y_prob_full[y == 1], bins=60, alpha=0.6, color='#e74c3c', label='TARGET=1 (default)', density=True)
ax.set_xlabel('Probabilidad predicha'); ax.set_ylabel('Densidad')
ax.set_title('Distribución de scores — Red Neuronal')
ax.legend()
plt.tight_layout()
plt.show()

---
## Predicciones Test

In [ ]:
y_pred_test = model_final.predict(X_test_scaled, verbose=0).ravel()
submission = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_pred_test})
_sub_path = DATA_DIR / 'submission_13_nn.csv'
submission.to_csv(_sub_path, index=False)
print(f'Submission guardada: {_sub_path.name}  ({len(submission):,} filas)')
print(f'TARGET stats: mean={y_pred_test.mean():.4f}  min={y_pred_test.min():.4f}  max={y_pred_test.max():.4f}')

---
## Resumen Final

In [ ]:
print('=' * 65)
print('RED NEURONAL (MLP + TensorFlow) — RESUMEN')
print('=' * 65)
print(f'  Arquitectura     : {best_p["n_layers"]} capas × {best_p["units"]} unidades')
print(f'  Dropout          : {best_p["dropout_rate"]:.3f}')
print(f'  Learning rate    : {best_p["learning_rate"]:.6f}')
print(f'  Batch size       : {best_p["batch_size"]}')
print(f'  Epochs (best)    : {metrics["n_epochs"]}')
print(f'  Train AUC        : {metrics["AUC"]}')
print(f'  Val AUC (20%)    : {metrics["Val_AUC"]}')
print(f'  Recall           : {metrics["Recall"]}')
print(f'  Precision        : {metrics["Precision"]}')
print(f'  F1               : {metrics["F1"]}')
print('=' * 65)

---
## Kaggle Submission — AUC Test (Public Leaderboard)

Envía el CSV a `home-credit-default-risk` y recupera el AUC del Public LB (~30% del test).

> **Límite**: 5 submissions/día.

In [ ]:
from kaggle import KaggleApi
import time

COMPETITION = 'home-credit-default-risk'
_msg = f'13_nn | layers={best_p["n_layers"]} units={best_p["units"]} | train={metrics["AUC"]:.4f} | val={metrics["Val_AUC"]:.4f}'

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub     = _as_str(getattr(matched, 'public_score',  None))
            status  = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(_sub_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, _sub_path.name, _msg)
print(f'\nAUC test  Public  LB : {public_auc}')
print(f'AUC test  Private LB : {private_auc}')